In [2]:
## Spline dataframe to stance and swing for one side
import os
from movedb.core import Trial
from movedb.file_io import sto_to_df
from scipy.interpolate import CubicSpline
import polars as pl
import numpy as np

output_dir = os.path.join('data', 'BAA', 'BAA01', 'Baseline')
pkl_path = os.path.join(output_dir, 'Walk05.pkl')
id_path = os.path.join(output_dir, 'Walk05_id.sto')

trial = Trial.from_pkl(pkl_path)

df, _ = sto_to_df(id_path)

phases = trial.get_stance_swing_phases(side='Left')

# Use the first phase for the splines
for i, phase in enumerate(phases):
    print(f"Phase {i}: {phase}")
phase = phases[0]
stance_phase = phase[:2]
swing_phase = phase[1:]
stance_times = [event.get_time(trial.points.rate) for event in stance_phase]
swing_times = [event.get_time(trial.points.rate) for event in swing_phase]

# Trim the dataframe to the stance and swing times
stance_df = df.filter(pl.col('time').is_between(stance_times[0], stance_times[-1]))
swing_df = df.filter(pl.col('time').is_between(swing_times[0], swing_times[-1]))

# Spline each to 100 points
num_points = 100
stance_times_np = stance_df['time'].to_numpy()
swing_times_np = swing_df['time'].to_numpy()
stance_times_spline = np.linspace(stance_times_np.min(), stance_times_np.max(), num_points)
swing_times_spline = np.linspace(swing_times_np.min(), swing_times_np.max(), num_points)

cs = CubicSpline(stance_times_np, stance_df.drop('time').to_numpy(), axis=0)
stance_spline = cs(stance_times_spline)
cs = CubicSpline(swing_times_np, swing_df.drop('time').to_numpy(), axis=0)
swing_spline = cs(swing_times_spline)

# Create a new dataframe with the splined data
stance_spline_df = pl.DataFrame({
    'time': stance_times_spline,
    **{f'{col}': stance_spline[:, i] for i, col in enumerate(stance_df.columns[1:])}
})
swing_spline_df = pl.DataFrame({
    'time': swing_times_spline,
    **{f'{col}': swing_spline[:, i] for i, col in enumerate(swing_df.columns[1:])}
})
# Concatenate the two dataframes vertically so that the time column is monotonically increasing
spline_df = pl.concat([stance_spline_df, swing_spline_df]).sort('time')

# Plot the splined data
spline_df.plot.line(x='time', y='ankle_l_flx_moment')


Phase 0: (Event(label='Foot Strike', context='Left', frame=None, time=3.055000066757202, description='The instant the heel strikes the ground'), Event(label='Foot Off', context='Left', frame=None, time=3.3299999237060547, description='The instant the toe leaves the ground'), Event(label='Foot Strike', context='Left', frame=None, time=3.450000047683716, description='The instant the heel strikes the ground'))


alt.Chart(...)

In [ ]:
# Pathology group subjects
NR = [
    "LGS09", "LGS10", "LGS13", "LGS14", "E17", "E20", "E21", "LGS11", "LGS12", "LGS15", "LGS16", 
    "T05", "T06", "E16", "E18", "E19", "LGS08", "T08", "E08", "E12", "E13", "E14", "E15", "E26", 
    "LGS05", "LGS06", "LGS07", "T07"
]
TEMR = [
    "T01", "T02", "T03", "T04", "T09", "T10", "T11", "T12", "G14_E23", "E09", "E10", 
    "A04", "A05", "A06", "A07", "A08"
]
HH = [
    "G01", "G02", "G03", "G04", "G05", "G06", "G07", "G08", "LGSH01", "LGSH02", "LGSH03", "LGSH04", 
    "G13", "G14", "G15", "G16", "G17", "G18"
]
TEMR_HH = [
    "H01", "H02", "H03", "H04", "H05", "H06", "H07", "H08", "G16_E24", "E02", "E06", "E11", 
    "A09", "A10", "A11", "A12"
]
HS = [
    "S01", "S03", "S04", "S05", "S06", "S07", "S08", "S13", "S09", "S10", "S11", "S12", 
    "S14", "S15", "S16", "S17"
]
TEMR_KG = [
    "K01", "K02", "K03", "K04", "K05", "K06", "K07", "K08", "G13_E22", "E01", "E03", "E04", "E05", 
    "A01", "A02", "A03"
]

pathology_groups = {
    "NR": NR,
    "TEMR": TEMR,
    "HH": HH,
    "TEMR_HH": TEMR_HH,
    "HS": HS,
    "TEMR_KG": TEMR_KG
}
